# TP3 – Retail Intelligence
## Étape 2 : Inventaire des sources

In [1]:
## Préparer Spark et les chemins

import os
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("TP3-Retail-Intelligence_Inventaire")
    .master("local[*]")
    .config("spark.hadoop.fs.defaultFS", "file:///")
    .getOrCreate()
)

PROJECT_ROOT = os.path.expanduser("~/retail-intelligence/tp3")
DATA_ROOT = os.path.join(PROJECT_ROOT, "data")

fichier_ventes = os.path.join(
     DATA_ROOT, "bronze", "store_sales", "store_sales.dat"
)

fichier_stocks = os.path.join(
    DATA_ROOT, "bronze", "inventory", "inventory.dat"
)

print("Fichier ventes :", fichier_ventes)
print("Fichier stocks :", fichier_stocks)

print("Fichier ventes présent :", os.path.isfile(fichier_ventes))
print("Fichier stocks présent :", os.path.isfile(fichier_stocks))

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/19 00:09:46 WARN Utils: Your hostname, NAVID-DELL, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/19 00:09:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 00:09:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Fichier ventes : /home/navid/retail-intelligence/tp3/data/bronze/store_sales/store_sales.dat
Fichier stocks : /home/navid/retail-intelligence/tp3/data/bronze/inventory/inventory.dat
Fichier ventes présent : True
Fichier stocks présent : True


In [2]:
## Définir les schémas

schema_ventes = StructType([
    StructField("ss_sold_date_sk", LongType(), True),
    StructField("ss_sold_time_sk", LongType(), True),
    StructField("ss_item_sk", LongType(), False),
    StructField("ss_customer_sk", LongType(), True),
    StructField("ss_cdemo_sk", LongType(), True),
    StructField("ss_hdemo_sk", LongType(), True),
    StructField("ss_addr_sk", LongType(), True),
    StructField("ss_store_sk", LongType(), True),
    StructField("ss_promo_sk", LongType(), True),
    StructField("ss_ticket_number", LongType(), False),
    StructField("ss_quantity", IntegerType(), True),
    StructField("ss_wholesale_cost", DecimalType(7, 2), True),
    StructField("ss_list_price", DecimalType(7, 2), True),
    StructField("ss_sales_price", DecimalType(7, 2), True),
    StructField("ss_ext_discount_amt", DecimalType(7, 2), True),
    StructField("ss_ext_sales_price", DecimalType(7, 2), True),
    StructField("ss_ext_wholesale_cost", DecimalType(7, 2), True),
    StructField("ss_ext_list_price", DecimalType(7, 2), True),
    StructField("ss_ext_tax", DecimalType(7, 2), True),
    StructField("ss_coupon_amt", DecimalType(7, 2), True),
    StructField("ss_net_paid", DecimalType(7, 2), True),
    StructField("ss_net_paid_inc_tax", DecimalType(7, 2), True),
    StructField("ss_net_profit", DecimalType(7, 2), True)
])

schema_stocks = StructType([
    StructField("inv_date_sk", LongType(), False),
    StructField("inv_item_sk", LongType(), False),
    StructField("inv_warehouse_sk", LongType(), False),
    StructField("inv_quantity_on_hand", IntegerType(), True)
])

print("Schéma ventes défini :", len(schema_ventes.fields), "colonnes")
print("Schéma stocks défini :", len(schema_stocks.fields), "colonnes")

Schéma ventes défini : 23 colonnes
Schéma stocks défini : 4 colonnes


In [3]:
## Lire les fichiers DAT

df_ventes = (
    spark.read
    .option("header", False)
    .option("sep", "|")
    .option("mode", "PERMISSIVE")
    .schema(schema_ventes)
    .csv(fichier_ventes)
)

df_stocks = (
    spark.read
    .option("header", False)
    .option("sep", "|")
    .option("mode", "PERMISSIVE")
    .schema(schema_stocks)
    .csv(fichier_stocks)
)

print("Chargement des fichiers terminé.")

Chargement des fichiers terminé.


In [4]:
## Contrôle des volumes

print("Nombre de ventes :", df_ventes.count())
print("Nombre de stocks :", df_stocks.count())

Nombre de ventes : 667579


[Stage 3:=========>                                               (2 + 10) / 12]

Nombre de stocks : 23255100


In [5]:
## Contrôle des schémas

print("SCHEMA DES VENTES")
df_ventes.printSchema()

print("SCHEMA DES STOCKS")
df_stocks.printSchema()

SCHEMA DES VENTES
root
 |-- ss_sold_date_sk: long (nullable = true)
 |-- ss_sold_time_sk: long (nullable = true)
 |-- ss_item_sk: long (nullable = true)
 |-- ss_customer_sk: long (nullable = true)
 |-- ss_cdemo_sk: long (nullable = true)
 |-- ss_hdemo_sk: long (nullable = true)
 |-- ss_addr_sk: long (nullable = true)
 |-- ss_store_sk: long (nullable = true)
 |-- ss_promo_sk: long (nullable = true)
 |-- ss_ticket_number: long (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_list_price: decimal(7,2) (nullable = true)
 |-- ss_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_discount_amt: decimal(7,2) (nullable = true)
 |-- ss_ext_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_ext_list_price: decimal(7,2) (nullable = true)
 |-- ss_ext_tax: decimal(7,2) (nullable = true)
 |-- ss_coupon_amt: decimal(7,2) (nullable = true)
 |-- ss_net_paid: decimal

In [6]:
## Observation des 1ères lignes

print("EXTRAIT DES VENTES")
df_ventes.show(5, truncate=False)

print("EXTRAIT DES STOCKS")
df_stocks.show(5, truncate=False)

EXTRAIT DES VENTES
+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+--------------+-------------------+------------------+---------------------+-----------------+----------+-------------+-----------+-------------------+-------------+
|ss_sold_date_sk|ss_sold_time_sk|ss_item_sk|ss_customer_sk|ss_cdemo_sk|ss_hdemo_sk|ss_addr_sk|ss_store_sk|ss_promo_sk|ss_ticket_number|ss_quantity|ss_wholesale_cost|ss_list_price|ss_sales_price|ss_ext_discount_amt|ss_ext_sales_price|ss_ext_wholesale_cost|ss_ext_list_price|ss_ext_tax|ss_coupon_amt|ss_net_paid|ss_net_paid_inc_tax|ss_net_profit|
+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+--------------+-------------------+------------------+---------------------+-----------------+----------+-------

In [7]:
## Recherche de valeurs nulles

def controle_nulls(df, nom):
    print(f"Valeurs nulles - {nom}")

    expressions = [
        F.count(
            F.when(F.col(c).isNull(), c)
        ).alias(c)
        for c in df.columns
    ]

    df.select(expressions).show(truncate=False)


controle_nulls(df_ventes, "VENTES")
controle_nulls(df_stocks, "STOCKS")

Valeurs nulles - VENTES


+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+--------------+-------------------+------------------+---------------------+-----------------+----------+-------------+-----------+-------------------+-------------+
|ss_sold_date_sk|ss_sold_time_sk|ss_item_sk|ss_customer_sk|ss_cdemo_sk|ss_hdemo_sk|ss_addr_sk|ss_store_sk|ss_promo_sk|ss_ticket_number|ss_quantity|ss_wholesale_cost|ss_list_price|ss_sales_price|ss_ext_discount_amt|ss_ext_sales_price|ss_ext_wholesale_cost|ss_ext_list_price|ss_ext_tax|ss_coupon_amt|ss_net_paid|ss_net_paid_inc_tax|ss_net_profit|
+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+--------------+-------------------+------------------+---------------------+-----------------+----------+-------------+-----------+

[Stage 11:==============>                                          (3 + 9) / 12]

+-----------+-----------+----------------+--------------------+
|inv_date_sk|inv_item_sk|inv_warehouse_sk|inv_quantity_on_hand|
+-----------+-----------+----------------+--------------------+
|0          |0          |0               |5999                |
+-----------+-----------+----------------+--------------------+



In [8]:
## Vérification de clés candidates

# Clé candidate des ventes
doublons_ventes = (
    df_ventes
    .groupBy("ss_item_sk", "ss_ticket_number")
    .count()
    .filter(F.col("count") > 1)
)

print("Doublons sur la clé candidate ventes :",
      doublons_ventes.count())


# Clé candidate des stocks
doublons_stocks = (
    df_stocks
    .groupBy(
        "inv_date_sk",
        "inv_item_sk",
        "inv_warehouse_sk"
    )
    .count()
    .filter(F.col("count") > 1)
)

print("Doublons sur la clé candidate stocks :",
      doublons_stocks.count())

Doublons sur la clé candidate ventes : 4038


[Stage 22:>                                                       (0 + 12) / 13]

[53.059s][warning][gc,alloc] Executor task launch worker for task 10.0 in stage 22.0 (TID 95): Retried waiting for GCLocker too often allocating 2097154 words


26/11/19 00:10:37 WARN TaskMemoryManager: Failed to allocate a page (16777216 bytes), try spilling task memory.
java.lang.OutOfMemoryError: Java heap space
	at org.apache.spark.unsafe.memory.HeapMemoryAllocator.allocate(HeapMemoryAllocator.java:72)
	at org.apache.spark.memory.TaskMemoryManager.allocatePageInternal(TaskMemoryManager.java:568)
	at org.apache.spark.memory.TaskMemoryManager.allocatePage(TaskMemoryManager.java:505)
	at org.apache.spark.memory.MemoryConsumer.allocateArray(MemoryConsumer.java:99)
	at org.apache.spark.unsafe.map.BytesToBytesMap.allocate(BytesToBytesMap.java:876)
	at org.apache.spark.unsafe.map.BytesToBytesMap.growAndRehash(BytesToBytesMap.java:999)
	at org.apache.spark.unsafe.map.BytesToBytesMap$Location.append(BytesToBytesMap.java:817)
	at org.apache.spark.sql.execution.UnsafeFixedWidthAggregationMap.getAggregationBufferFromUnsafeRow(UnsafeFixedWidthAggregationMap.java:135)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$GeneratedIteratorForCodeg

Doublons sur la clé candidate stocks : 0


In [9]:
## Contrôle des valeurs négatives

print("Quantités de ventes négatives :",
      df_ventes.filter(F.col("ss_quantity") < 0).count())

print("Quantités de stock négatives :",
      df_stocks.filter(F.col("inv_quantity_on_hand") < 0).count())

Quantités de ventes négatives : 0


[Stage 29:======================================>                  (8 + 4) / 12]

Quantités de stock négatives : 0


In [10]:
## Recherche d'une clé candidate plus complète

# Tester une combinaison plus complète
cle_ventes = [
    "ss_sold_date_sk",
    "ss_store_sk",
    "ss_ticket_number",
    "ss_item_sk"
]

doublons_ventes_complets = (
    df_ventes
    .groupBy(*cle_ventes)
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Groupes en doublon avec la clé complète :",
    doublons_ventes_complets.count()
)

doublons_ventes_complets.show(10, truncate=False)

Groupes en doublon avec la clé complète : 4038


[Stage 38:=======================>                                 (5 + 7) / 12]

+---------------+-----------+----------------+----------+-----+
|ss_sold_date_sk|ss_store_sk|ss_ticket_number|ss_item_sk|count|
+---------------+-----------+----------------+----------+-----+
|36921          |11         |1567            |10000     |2    |
|36923          |6          |1649            |10563     |2    |
|36964          |9          |3689            |10003     |2    |
|36916          |9          |1305            |10002     |2    |
|37036          |5          |7230            |7628      |2    |
|36931          |11         |2069            |12573     |2    |
|37015          |5          |6181            |8601      |2    |
|36943          |11         |2659            |10000     |2    |
|36925          |0          |1752            |10001     |2    |
|36944          |5          |2673            |7628      |2    |
+---------------+-----------+----------------+----------+-----+
only showing top 10 rows


In [11]:
# Nombre de lignes appartenant aux groupes en doublon
lignes_doublonnees = (
    df_ventes
    .join(
        doublons_ventes,
        ["ss_item_sk", "ss_ticket_number"],
        "inner"
    )
)

print(
    "Nombre de lignes appartenant aux groupes en doublon :",
    lignes_doublonnees.count()
)

[Stage 45:>                                                       (0 + 12) / 12]

Nombre de lignes appartenant aux groupes en doublon : 8174


In [12]:
def taux_nulls(df):
    total = df.count()

    for colonne in df.columns:
        nb_nulls = (
            df.filter(F.col(colonne).isNull()).count()
        )

        pourcentage = (nb_nulls / total) * 100

        print(
            f"{colonne}: {nb_nulls} valeurs nulles "
            f"({pourcentage:.4f} %)"
        )

taux_nulls(df_ventes)
taux_nulls(df_stocks)

ss_sold_date_sk: 0 valeurs nulles (0.0000 %)
ss_sold_time_sk: 0 valeurs nulles (0.0000 %)
ss_item_sk: 0 valeurs nulles (0.0000 %)
ss_customer_sk: 0 valeurs nulles (0.0000 %)
ss_cdemo_sk: 238 valeurs nulles (0.0357 %)
ss_hdemo_sk: 295 valeurs nulles (0.0442 %)
ss_addr_sk: 448 valeurs nulles (0.0671 %)
ss_store_sk: 0 valeurs nulles (0.0000 %)
ss_promo_sk: 158 valeurs nulles (0.0237 %)
ss_ticket_number: 0 valeurs nulles (0.0000 %)
ss_quantity: 0 valeurs nulles (0.0000 %)
ss_wholesale_cost: 0 valeurs nulles (0.0000 %)
ss_list_price: 0 valeurs nulles (0.0000 %)
ss_sales_price: 0 valeurs nulles (0.0000 %)
ss_ext_discount_amt: 0 valeurs nulles (0.0000 %)
ss_ext_sales_price: 0 valeurs nulles (0.0000 %)
ss_ext_wholesale_cost: 0 valeurs nulles (0.0000 %)
ss_ext_list_price: 0 valeurs nulles (0.0000 %)
ss_ext_tax: 0 valeurs nulles (0.0000 %)
ss_coupon_amt: 0 valeurs nulles (0.0000 %)
ss_net_paid: 0 valeurs nulles (0.0000 %)
ss_net_paid_inc_tax: 0 valeurs nulles (0.0000 %)
ss_net_profit: 0 valeurs 

inv_date_sk: 0 valeurs nulles (0.0000 %)


inv_item_sk: 0 valeurs nulles (0.0000 %)


inv_warehouse_sk: 0 valeurs nulles (0.0000 %)


[Stage 134:==============>                                         (3 + 9) / 12]

inv_quantity_on_hand: 5999 valeurs nulles (0.0258 %)


In [13]:
colonnes_financieres = [
    "ss_wholesale_cost",
    "ss_list_price",
    "ss_sales_price",
    "ss_ext_discount_amt",
    "ss_ext_sales_price",
    "ss_ext_wholesale_cost",
    "ss_ext_list_price",
    "ss_ext_tax",
    "ss_coupon_amt",
    "ss_net_paid",
    "ss_net_paid_inc_tax",
    "ss_net_profit"
]

expressions = []

for colonne in colonnes_financieres:
    expressions.extend([
        F.min(F.col(colonne)).alias(f"{colonne}_min"),
        F.max(F.col(colonne)).alias(f"{colonne}_max")
    ])

df_ventes.select(*expressions).show(truncate=False)

[Stage 137:==========================================>             (9 + 3) / 12]

+---------------------+---------------------+-----------------+-----------------+------------------+------------------+-----------------------+-----------------------+----------------------+----------------------+-------------------------+-------------------------+---------------------+---------------------+--------------+--------------+-----------------+-----------------+---------------+---------------+-----------------------+-----------------------+-----------------+-----------------+
|ss_wholesale_cost_min|ss_wholesale_cost_max|ss_list_price_min|ss_list_price_max|ss_sales_price_min|ss_sales_price_max|ss_ext_discount_amt_min|ss_ext_discount_amt_max|ss_ext_sales_price_min|ss_ext_sales_price_max|ss_ext_wholesale_cost_min|ss_ext_wholesale_cost_max|ss_ext_list_price_min|ss_ext_list_price_max|ss_ext_tax_min|ss_ext_tax_max|ss_coupon_amt_min|ss_coupon_amt_max|ss_net_paid_min|ss_net_paid_max|ss_net_paid_inc_tax_min|ss_net_paid_inc_tax_max|ss_net_profit_min|ss_net_profit_max|
+---------------

In [14]:
print("Analyse de ventes les plus déficitaires:")

df_ventes.select(
    "ss_sold_date_sk",
    "ss_store_sk",
    "ss_item_sk",
    "ss_quantity",
    "ss_sales_price",
    "ss_ext_sales_price",
    "ss_net_paid",
    "ss_net_profit"
).orderBy(
    F.col("ss_net_profit").asc()
).show(10, truncate=False)

print("écarts entre les montants :")
df_ventes.select(
    "ss_quantity",
    "ss_ext_sales_price",
    "ss_ext_wholesale_cost",
    "ss_ext_discount_amt",
    "ss_coupon_amt",
    "ss_net_paid",
    "ss_net_profit"
).orderBy(
    F.col("ss_net_profit").asc()
).show(10, truncate=False)

print("Cohérences des montants financiers:")
df_ventes.select(
    F.count("*").alias("total_lignes"),

    F.sum(
        F.when(
            F.col("ss_net_profit") !=
            (F.col("ss_net_paid") - F.col("ss_ext_wholesale_cost")),
            1
        ).otherwise(0)
    ).alias("ecarts_benefice"),

    F.sum(
        F.when(
            F.col("ss_net_paid") >
            F.col("ss_ext_sales_price"),
            1
        ).otherwise(0)
    ).alias("net_superieur_ventes"),

    F.sum(
        F.when(
            F.col("ss_sales_price") == 0,
            1
        ).otherwise(0)
    ).alias("prix_vente_nul"),

    F.sum(
        F.when(
            F.col("ss_net_paid") == 0,
            1
        ).otherwise(0)
    ).alias("montant_net_nul")
).show(truncate=False)

Analyse de ventes les plus déficitaires:


+---------------+-----------+----------+-----------+--------------+------------------+-----------+-------------+
|ss_sold_date_sk|ss_store_sk|ss_item_sk|ss_quantity|ss_sales_price|ss_ext_sales_price|ss_net_paid|ss_net_profit|
+---------------+-----------+----------+-----------+--------------+------------------+-----------+-------------+
|37257          |1          |10003     |100        |7.04          |704.00            |70.40      |-9866.60     |
|37505          |2          |10002     |99         |0.00          |0.00              |0.00       |-9802.98     |
|38084          |6          |13546     |98         |4.20          |411.60            |4.12       |-9730.22     |
|37952          |8          |16561     |100        |3.61          |361.00            |198.55     |-9720.45     |
|37379          |8          |17405     |100        |1.01          |101.00            |101.00     |-9688.00     |
|37073          |1          |10579     |97         |25.80         |2502.60           |0.00      

+-----------+------------------+---------------------+-------------------+-------------+-----------+-------------+
|ss_quantity|ss_ext_sales_price|ss_ext_wholesale_cost|ss_ext_discount_amt|ss_coupon_amt|ss_net_paid|ss_net_profit|
+-----------+------------------+---------------------+-------------------+-------------+-----------+-------------+
|100        |704.00            |9937.00              |11022.00           |633.60       |70.40      |-9866.60     |
|99         |0.00              |9802.98              |12352.23           |0.00         |0.00       |-9802.98     |
|98         |411.60            |9734.34              |13314.28           |407.48       |4.12       |-9730.22     |
|100        |361.00            |9919.00              |17692.00           |162.45       |198.55     |-9720.45     |
|100        |101.00            |9789.00              |9982.00            |0.00         |101.00     |-9688.00     |
|97         |2502.60           |9684.48              |12218.12           |2502.6

[Stage 142:==============>                                         (3 + 9) / 12]

+------------+---------------+--------------------+--------------+---------------+
|total_lignes|ecarts_benefice|net_superieur_ventes|prix_vente_nul|montant_net_nul|
+------------+---------------+--------------------+--------------+---------------+
|667579      |0              |0                   |3324          |3954           |
+------------+---------------+--------------------+--------------+---------------+



In [15]:
## Fermeture de session

spark.stop()